## IMPORT

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
import sklearn
from datetime import datetime
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import joblib
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV

## Modèle 1 : Prédiction de la taille d'un panier pour un acheteur d'alcool

In [32]:
df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")

#/-------------------------------------------------------\#
#|------------- Tri et création de colonnes -------------|#
#\-------------------------------------------------------/#

# Récupère si l'article dans le panier est de l'alcool
df["is_alcohol"] = df["department"].str.contains("alcohol", case=False, na=False).astype(int)

# Création d'un nouveau dataset en récupérant la taille de la commande des clients en fonction de l'id de la commande
basket_size = df.groupby("order_id").size().reset_index(name="basket_size")

# Création d'un nouveau dataset en récupérant les données en fonction de l'id de la commande qui me permettra de trier plus tard
orders = df.groupby("order_id").agg(
    user_id=("user_id", "first"),
    order_number=("order_number", "first"),
    order_hour_of_day=("order_hour_of_day", "first"),
    order_dow=("order_dow", "first"),
    days_since_prior_order=("days_since_prior_order", "first"),
    alcohol_count=("is_alcohol", "sum"),
    reorder_count=("reordered", "sum"),
).reset_index()

# Ce merge là sert à trier une nouvelle fois en fusionnant avec le datasets basket_size en fonction de l'id de la commande
orders = orders.merge(basket_size, on="order_id")

# Récupère le nombre d'article d'alcool en fonction de la taille du panier
orders["alcohol_ratio"] = orders["alcohol_count"]/orders["basket_size"]

# Récupère le nombre d'article commendés plusieurs fois en fonction de la taille du panier
orders["reorder_ratio"]  = orders["reorder_count"]/orders["basket_size"]

# Récupère si on est le week-end en récupérant à partir de la colonnes order_dow en prennant uniquement les valeurs entre 6 et 0 (donc Samedi et Dimanche)
orders["is_weekend"] = orders["order_dow"].isin([6, 0]).astype(int)
orders["time_of_day"] = pd.cut(
    orders["order_hour_of_day"],
    bins=[0, 6, 12, 17, 21],
    labels=["nuit", "matin", "apres_midi", "soir"],
    right=False
).astype(str)

#/--------------------------------------------------------------\#
#|------------------------ Entraînement ------------------------|#
#\--------------------------------------------------------------/#

# Ici je définis les colonnes que le modèle doit utiliser pour s'entraîner
feature_cols = [
    "order_hour_of_day", "order_dow",
    "is_weekend", "time_of_day",
    "days_since_prior_order", "order_number",
    "alcohol_count", "alcohol_ratio",
    "reorder_count", "reorder_ratio",
]

# Transforme les chaînes de charactère en True/False et définis mon X
X = pd.get_dummies(orders[feature_cols])

# Récupère le y, la target
y = orders["basket_size"]

# Split les données entre tests et entraînements 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.05, random_state=42
)

# Définis le modèle utiliser et ses hyperparamètres
model = HistGradientBoostingRegressor(
    max_iter=500,                 
    learning_rate=0.05,
    max_leaf_nodes=127,            
    min_samples_leaf=50,                    
    random_state=42,
)

# Lance l'entraînements du modèle
model.fit(X_train, y_train)

#/--------------------------------------------------------------\#
#|------------------------- Évaluation -------------------------|#
#\--------------------------------------------------------------/#

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE :{mae}")
print(f"RMSE :{rmse}")
print(f"R2 :{r2}")
print(orders)

KeyboardInterrupt: 

## Modèle 2 : Prédiction du contenu du panier

In [ ]:
df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")

grouped_by_orders = df.groupby(["order_id", "product_name"]).agg({
    "user_id": "first",
    "order_number": "first",
    "order_dow": "first",
    "order_hour_of_day": "first",
    "order_number": "first",
    "days_since_prior_order": "first",
    
}).reset_index()

grouped_by_orders["size_of_the_basket"] = df.groupby("order_id")
print(grouped_by_orders)

# feature_cols = [
#     "order_number",
#     "order_dow",
#     "order_hour_of_day",
#     "days_since_prior_order",
#     "size_of_the_basket"
# ]

# X = pd.get_dummies(grouped_by_orders[feature_cols])
# y = grouped_by_orders["product_name"]

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# model = HistGradientBoostingRegressor(
#     max_iter=500,                 
#     learning_rate=0.05,
#     max_leaf_nodes=127,            
#     min_samples_leaf=50,                    
#     random_state=42,
# )

# model.fit(X_train, y_train)

# y_pred = model.predict(X_test)
# print(y_pred)

          order_id                              product_name  user_id  \
0                2  All Natural No Stir Creamy Almond Butter   202279   
1                2                                   Carrots   202279   
2                2                   Classic Blend Cole Slaw   202279   
3                2                            Coconut Butter   202279   
4                2                             Garlic Powder   202279   
...            ...                                       ...      ...   
12811675   2999999                     Chicken Tortilla Soup   100467   
12811676   2999999                      Organic Baby Spinach   100467   
12811677   2999999                     Organic Salted Butter   100467   
12811678   2999999                              Strawberries   100467   
12811679   2999999                              Variety Pack   100467   

          order_number  order_dow  order_hour_of_day  days_since_prior_order  \
0                    3          5          

## Modèle 3 : Prédire